In [1]:
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from huggingface_hub import login

In [2]:
path = Path.cwd()
data_dir = path.parent / "data"
preprocess_data = data_dir / "preprocess"

In [2]:
login(token=your_token)

In [3]:
from transformers import AutoTokenizer , AutoModel

model_name = "ai4bharat/IndicBERT-v3-4B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name,
    device_map="cuda",          # FORCES it completely into VRAM (no CPU split)
    torch_dtype=torch.float16,  # Cuts RAM/VRAM usage in half
    low_cpu_mem_usage=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/2.42k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/688 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/37.3k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights: 0it [00:00, ?it/s]

Gemma3Model LOAD REPORT from: ai4bharat/IndicBERT-v3-4B
Key                                                                         | Status     | 
----------------------------------------------------------------------------+------------+-
model.layers.{0...33}.mlp.up_proj.weight                                    | UNEXPECTED | 
model.layers.{0...33}.post_feedforward_layernorm.weight                     | UNEXPECTED | 
model.layers.{0...33}.post_attention_layernorm.weight                       | UNEXPECTED | 
model.layers.{0...33}.self_attn.k_norm.weight                               | UNEXPECTED | 
model.layers.{0...33}.mlp.down_proj.weight                                  | UNEXPECTED | 
model.layers.{0...33}.mlp.gate_proj.weight                                  | UNEXPECTED | 
model.layers.{0...33}.input_layernorm.weight                                | UNEXPECTED | 
model.layers.{0...33}.self_attn.k_proj.weight                               | UNEXPECTED | 
model.layers.{0...33}.se

In [3]:
data = pd.read_csv(preprocess_data / "indic-language.csv")

In [4]:
data.head()

,input,label
0,कनाडा अमेरिका और यूरोपीय संघ का अनुसरण करते हु...,Hindi
1,विदेशों में मूलधातुओं की कीमतों में कमजोरी के ...,Hindi
2,डेविड वॉर्नर पर क्रिकेट ऑस्ट्रेलिया ने 1 साल क...,Hindi
3,"अगर आपके पास फटे-पुराने नोट हैं, जिन्हें दुक...",Hindi
4,नोवेल लवासा ने देर रात बयान जारी कर कहा कि उन्...,Hindi


In [6]:
data["label"].value_counts()

,count
label,
Hindi,500
Gujarati,500
Punjabi,500
Marathi,500
Tamil,500
Telugu,500
Malayalam,500
Kannada,500


In [9]:
BATCH_SIZE = 32 
X_train = []
text = data["input"].to_list()

for i in tqdm(range(0, len(text), BATCH_SIZE)):
    
    batch_text = text[i : i + BATCH_SIZE]
    
    inputs = tokenizer(
        batch_text,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model(**inputs)


    batch_embeddings = outputs.last_hidden_state.mean(dim=1).float().cpu().numpy()
    X_train.extend(batch_embeddings)

X_train = np.array(X_train)
print("Final shape:", X_train.shape)

100%|██████████| 125/125 [00:13<00:00,  9.26it/s]

Final shape: (4000, 2560)


In [ ]:
X_train.shape

(14400, 2560)

In [5]:
embeddings_data = data_dir / "embeddings"

In [ ]:
np.save(embeddings_data / "indic_bert_v3.npy", X_train)